# ScreenAware - Model Training
This notebook trains the ML models for the ScreenAware app:
1. Risk classification (Low, Medium, High)
2. Mood rating regression
3. Screen usage clustering (KMeans)


In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from math import sqrt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error, mean_squared_error, silhouette_score
from xgboost import XGBClassifier, XGBRegressor
from sklearn.cluster import KMeans


## 1. Load data


In [2]:
# Load dataset
df = pd.read_csv("data/digital_diet_mental_health.csv")
print("Dataset shape:", df.shape)
df.head(3)


ERROR: [Errno 2] No such file or directory: 'data/digital_diet_mental_health.csv'


## 2. Risk level calculation
We compute a digital strain score based on:
- Screen-to-sleep ratio (displacement)
- Stress and poor sleep quality
- Inactivity ratio (screen time vs physical activity)


In [3]:
# Calculate component scores
disp = df["daily_screen_time_hours"] / (df["sleep_duration_hours"] + 0.1)
score_disp = np.clip((disp - 0.4) / (1.6 - 0.4) * 10, 0, 10)

score_stress_sleep = (df["stress_level"] + (11 - df["sleep_quality"])) / 2

score_inact = np.clip(
    (df["daily_screen_time_hours"] / (df["physical_activity_hours_per_week"] + 1)) * 2,
    0,
    10
)

# Combined strain index (0 to 10)
df["cdsi"] = 0.40 * score_disp + 0.40 * score_stress_sleep + 0.20 * score_inact

def assign_risk(score):
    if score >= 6.2:
        return "High"
    elif score <= 4.2:
        return "Low"
    else:
        return "Medium"

df["risk_level"] = df["cdsi"].apply(assign_risk)

print("Risk distribution:")
print(df["risk_level"].value_counts())


ERROR: name 'df' is not defined


## 3. Train risk classifier (XGBoost)
Features used: screen time, sleep hours, stress level, sleep quality, and physical activity.


In [4]:
clf_features = [
    "daily_screen_time_hours",
    "sleep_duration_hours",
    "stress_level",
    "sleep_quality",
    "physical_activity_hours_per_week"
]

X_clf = df[clf_features]
y_clf = df["risk_level"]

le = LabelEncoder()
y_clf_encoded = le.fit_transform(y_clf)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf_encoded, test_size=0.2, random_state=42, stratify=y_clf_encoded
)

scaler_clf = StandardScaler()
X_train_c_scaled = scaler_clf.fit_transform(X_train_c)
X_test_c_scaled = scaler_clf.transform(X_test_c)

clf_model = XGBClassifier(
    n_estimators=180,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    eval_metric="mlogloss"
)
clf_model.fit(X_train_c_scaled, y_train_c)

# Evaluate on test set
y_pred_c = clf_model.predict(X_test_c_scaled)
print(f"Test Accuracy: {accuracy_score(y_test_c, y_pred_c):.4f}
")
print(classification_report(y_test_c, y_pred_c, target_names=le.classes_))


ERROR: unterminated f-string literal (detected at line 36) (<string>, line 36)


## 4. Mood rating regression
We add interaction features (`screen_sleep_ratio`, `stress_x_sleep`, `activity_balance`, `wellness_score`) and train an XGBRegressor.


In [5]:
# Feature engineering
df["screen_sleep_ratio"] = df["daily_screen_time_hours"] / (df["sleep_duration_hours"] + 1)
df["stress_x_sleep"] = df["stress_level"] * df["sleep_quality"]
df["activity_balance"] = df["physical_activity_hours_per_week"] / (df["daily_screen_time_hours"] + 1)
df["wellness_score"] = (
    (df["sleep_quality"] + df["physical_activity_hours_per_week"]) / 2
    - (df["stress_level"] + df["daily_screen_time_hours"] / 2)
)

reg_features = clf_features + [
    "screen_sleep_ratio",
    "stress_x_sleep",
    "activity_balance",
    "wellness_score"
]

X_reg = df[reg_features]
y_reg = df["mood_rating"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

scaler_reg = StandardScaler()
X_train_r_scaled = scaler_reg.fit_transform(X_train_r)
X_test_r_scaled = scaler_reg.transform(X_test_r)

reg_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
reg_model.fit(X_train_r_scaled, y_train_r)

y_pred_r = reg_model.predict(X_test_r_scaled)
print(f"Regression MAE : {mean_absolute_error(y_test_r, y_pred_r):.4f}")
print(f"Regression RMSE: {sqrt(mean_squared_error(y_test_r, y_pred_r)):.4f}")

# Get min/max bounds for 1-5 mapping
all_preds = reg_model.predict(scaler_reg.transform(X_reg))
mood_min_range = float(np.percentile(all_preds, 1))
mood_max_range = float(np.percentile(all_preds, 99))
print(f"Mood range bounds: [{mood_min_range:.2f}, {mood_max_range:.2f}]")


ERROR: name 'df' is not defined


## 5. Screen habits clustering (KMeans)
Cluster users based on social media, gaming, entertainment, and work hours.


In [6]:
cluster_features = [
    "social_media_hours",
    "gaming_hours",
    "entertainment_hours",
    "work_related_hours"
]

X_cluster = df[cluster_features]
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

km = KMeans(n_clusters=5, random_state=42, n_init=10)
labels = km.fit_predict(X_cluster_scaled)
print(f"KMeans Silhouette Score: {silhouette_score(X_cluster_scaled, labels):.4f}")

# Label clusters by highest average category
centers = scaler_cluster.inverse_transform(km.cluster_centers_)
centers_df = pd.DataFrame(centers, columns=cluster_features)

cluster_name_map = {}
for idx, row in centers_df.iterrows():
    dom = row.idxmax()
    if dom == "social_media_hours":
        cluster_name_map[idx] = "Social Media Dominant"
    elif dom == "gaming_hours":
        cluster_name_map[idx] = "Gaming Heavy"
    elif dom == "entertainment_hours":
        cluster_name_map[idx] = "Entertainment Focused" if row["work_related_hours"] >= 1.0 else "Casual Leisure"
    elif dom == "work_related_hours":
        cluster_name_map[idx] = "Work Oriented"
    else:
        cluster_name_map[idx] = "Balanced Usage"

print("
Cluster personas:")
for idx, name in cluster_name_map.items():
    print(f"  Cluster {idx}: {name}")


ERROR: unterminated string literal (detected at line 34) (<string>, line 34)


## 6. Save models and artifacts


In [7]:
models_dir = "models"
backend_models_dir = os.path.abspath(os.path.join("..", "backend", "app", "models"))
os.makedirs(models_dir, exist_ok=True)
os.makedirs(backend_models_dir, exist_ok=True)

artifacts = {
    "clf_features": clf_features,
    "reg_features": reg_features,
    "cluster_features": cluster_features,
    "best_cluster_method": "KMeans(k=5)",
    "cluster_name_map": cluster_name_map,
    "clf_target_labels": list(le.classes_),
    "mood_min_range": mood_min_range,
    "mood_max_range": mood_max_range
}

files = {
    "risk_model.pkl": clf_model,
    "scaler_clf.pkl": scaler_clf,
    "label_encoder_clf.pkl": le,
    "mood_model.pkl": reg_model,
    "scaler_reg.pkl": scaler_reg,
    "cluster_model.pkl": km,
    "scaler_cluster.pkl": scaler_cluster,
    "artifacts.pkl": artifacts
}

for name, obj in files.items():
    joblib.dump(obj, os.path.join(models_dir, name))
    joblib.dump(obj, os.path.join(backend_models_dir, name))
    print(f"Saved {name}")

print("
All models exported successfully.")


ERROR: unterminated string literal (detected at line 33) (<string>, line 33)


## 7. Quick sample test


In [8]:
test_sample = {
    "daily_screen_time_hours": 3.0,
    "sleep_duration_hours": 8.0,
    "stress_level": 2.0,
    "sleep_quality": 8.0,
    "physical_activity_hours_per_week": 5.0,
    "social_media_hours": 1.0,
    "gaming_hours": 0.5,
    "entertainment_hours": 1.0,
    "work_related_hours": 0.5
}

test_df = pd.DataFrame([test_sample])

# Predict risk
X_r = scaler_clf.transform(test_df[clf_features])
risk_pred = le.inverse_transform(clf_model.predict(X_r))[0]

# Predict mood
tdf_m = test_df.copy()
tdf_m["screen_sleep_ratio"] = tdf_m["daily_screen_time_hours"] / (tdf_m["sleep_duration_hours"] + 1)
tdf_m["stress_x_sleep"] = tdf_m["stress_level"] * tdf_m["sleep_quality"]
tdf_m["activity_balance"] = tdf_m["physical_activity_hours_per_week"] / (tdf_m["daily_screen_time_hours"] + 1)
tdf_m["wellness_score"] = (
    (tdf_m["sleep_quality"] + tdf_m["physical_activity_hours_per_week"]) / 2
    - (tdf_m["stress_level"] + tdf_m["daily_screen_time_hours"] / 2)
)
X_m = scaler_reg.transform(tdf_m[reg_features])
raw_mood = reg_model.predict(X_m)[0]
scaled_mood = int(round(np.clip(1 + (raw_mood - mood_min_range) * 4 / (mood_max_range - mood_min_range), 1, 5)))

# Predict cluster
X_c = scaler_cluster.transform(test_df[cluster_features])
c_id = km.predict(X_c)[0]
c_name = cluster_name_map.get(c_id, "Unknown")

print("Sample Prediction Results:")
print("  Risk Level   :", risk_pred)
print("  Mood Rating  :", f"{scaled_mood}/5 (raw: {raw_mood:.2f})")
print("  Usage Persona:", c_name)


ERROR: name 'scaler_clf' is not defined
